# HIPC scRNA-seq Annotation Benchmark — Team 11
Fred Hutchinson Cancer Center, McElrath/Moyo Lab, VIDD  
Submission deadline: July 12, 2026

## 1. Setup

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import harmonypy as hm
import celltypist
from celltypist import models
import scrublet as scr
import mygene
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from scipy import stats

os.chdir('/fh/fast/mcelrath_j/McElrath_NGS/C_Marini_output/HIPC_Challenge/team11/recipe_data/')

# Load CellTypist model
models.download_models(model='Immune_All_Low.pkl')
model_ct = models.Model.load('Immune_All_Low.pkl')

## 2. Label Mapping Dictionaries

In [ ]:
label_mapping = {
    'Tcm/Naive helper T cells':                          'CD4 Naive / T Central Memory',
    'Tem/Effector helper T cells':                       'CD4 T Effector Memory',
    'Type 1 helper T cells':                             'CD4 T Effector Memory',
    'Type 17 helper T cells':                            'CD4 T Effector Memory',
    'Follicular helper T cells':                         'CD4 T Effector Memory',
    'Regulatory T cells':                                'Treg',
    'Tcm/Naive cytotoxic T cells':                       'CD8 Naive / T Central Memory',
    'Tem/Temra cytotoxic T cells':                       'CD8 Cytotoxic / T Effector Memory',
    'Tem/Trm cytotoxic T cells':                         'CD8 Cytotoxic / T Effector Memory',
    'CD8a/a':                                            'CD8 Cytotoxic / T Effector Memory',
    'MAIT cells':                                        'MAIT Cell',
    'NKT cells':                                         'NKT Cell',
    'CRTAM+ gamma-delta T cells':                        'gdT Cell',
    'Cycling T cells':                                   'CD4 Naive / T Central Memory',
    'NK cells':                                          'NK Cell',
    'CD16+ NK cells':                                    'NK Cell',
    'CD16- NK cells':                                    'NK Cell',
    'Cycling NK cells':                                  'NK Cell',
    'Transitional NK':                                   'NK Cell',
    'Naive B cells':                                     'Naive B Cell',
    'Memory B cells':                                    'Memory B Cell',
    'Age-associated B cells':                            'Memory B Cell',
    'Transitional B cells':                              'Naive B Cell',
    'Plasma cells':                                      'Plasma Cell',
    'Plasmablasts':                                      'Plasmablast',
    'Proliferative germinal center B cells':             'Memory B Cell',
    'Classical monocytes':                               'Classical Monocyte',
    'Non-classical monocytes':                           'Non-Classical Monocyte',
    'Intermediate macrophages':                          'Intermediate Monocyte',
    'Monocytes':                                         'Classical Monocyte',
    'DC1':                                               'Conventional DC 1',
    'DC2':                                               'Conventional DC 2',
    'pDC':                                               'Plasmacytoid DC',
    'Migratory DCs':                                     'Conventional DC 1',
    'DC precursor':                                      'Conventional DC 1',
    'DC3':                                               'Conventional DC 2',
    'Mast cells':                                        'Mast Cell',
    'Megakaryocytes/platelets':                          'Platelet',
    'Megakaryocyte precursor':                           'Platelet',
    'Early MK':                                          'Platelet',
    'HSC/MPP':                                           'RBC',
    'Late erythroid':                                    'RBC',
    'Early erythroid':                                   'RBC',
    'Mid erythroid':                                     'RBC',
    'Erythrocytes':                                      'RBC',
    'Epithelial cells':                                  'T Cell',
    'Double-positive thymocytes':                        'T Cell',
    'Double-negative thymocytes':                        'T Cell',
    'ILC':                                               'NK Cell',
    'ILC3':                                              'NK Cell',
    'ILC precursor':                                     'NK Cell',
    'ETP':                                               'HSC',
    'ELP':                                               'HSC',
    'Early lymphoid/T lymphoid':                         'T Cell',
    'T(agonist)':                                        'T Cell',
    'Neutrophil-myeloid progenitor':                     'Myeloid Cell',
    'Myelocytes':                                        'Myeloid Cell',
    'MEMP':                                              'Myeloid Cell',
    'Megakaryocyte-erythroid-mast cell progenitor':      'Platelet',
    'Treg(diff)':                                        'Treg',
    'Mono-mac':                                          'Classical Monocyte',
    'MNP':                                               'Classical Monocyte',
    'Macrophages':                                       'Classical Monocyte',
}

parent_mapping = {
    'CD4 Naive / T Central Memory':      'CD4 T Cell (ab)',
    'CD4 T Effector Memory':             'CD4 T Cell (ab)',
    'Treg':                              'CD4 T Cell (ab)',
    'CD8 Naive / T Central Memory':      'CD8 T Cell (ab)',
    'CD8 Cytotoxic / T Effector Memory': 'CD8 T Cell (ab)',
    'gdT Cell':                          'T Cell',
    'MAIT Cell':                         'T Cell',
    'NKT Cell':                          'T Cell',
    'NK Cell':                           'Lymphoid Cell',
    'Naive B Cell':                      'B Cell',
    'Memory B Cell':                     'B Cell',
    'Plasma Cell':                       'Effector B',
    'Plasmablast':                       'Effector B',
    'Classical Monocyte':                'Monocyte',
    'Non-Classical Monocyte':            'Monocyte',
    'Intermediate Monocyte':             'Monocyte',
    'Conventional DC 1':                 'DC',
    'Conventional DC 2':                 'DC',
    'Plasmacytoid DC':                   'DC',
    'Neutrophil':                        'Granulocyte',
    'Eosinophil':                        'Granulocyte',
    'Basophil':                          'Granulocyte',
    'Mast Cell':                         'Granulocyte',
    'Platelet':                          'Blood Cell',
    'RBC':                               'Blood Cell',
    'HSC':                               'Blood Cell',
    'Doublet':                           'Blood Cell',
}

## 3. Infection Cohort (studies 01, 02, 06)

Study 07 excluded: pre-scaled data with negative values, no raw counts available.

In [ ]:
# Load unfiltered h5ad files
inf01 = sc.read_h5ad('infection_study_01/infection_study_01_unfiltered.h5ad')
inf02 = sc.read_h5ad('infection_study_02/infection_study_02_unfiltered.h5ad')
inf06 = sc.read_h5ad('infection_study_06/infection_study_06_unfiltered.h5ad')

# infection_study_06: convert Ensembl IDs to gene symbols
mg = mygene.MyGeneInfo()
ensembl_ids = inf06.var_names.tolist()
results = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human')
id_to_symbol = {r['query']: r['symbol'] for r in results if 'symbol' in r}
print(f'Mapped {len(id_to_symbol)} of {len(ensembl_ids)} genes')
new_names = [id_to_symbol.get(eid, eid) for eid in inf06.var_names.tolist()]
inf06.var_names = new_names
inf06.var_names_make_unique()

adatas_inf = {
    'infection_study_01': inf01,
    'infection_study_02': inf02,
    'infection_study_06': inf06,
}

In [ ]:
# QC per study
def filter_study(adata, name):
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
    n_before = adata.n_obs
    mt_thresh = 10 if '06' in name else 15
    adata = adata[
        (adata.obs['n_genes_by_counts'] >= 200) &
        (adata.obs['n_genes_by_counts'] <= 6000) &
        (adata.obs['total_counts'] < 50000) &
        (adata.obs['pct_counts_mt'] < mt_thresh)
    ].copy()
    print(f'{name}: {n_before} -> {adata.n_obs} cells after QC')
    return adata

for name in list(adatas_inf.keys()):
    adatas_inf[name] = filter_study(adatas_inf[name], name)

In [ ]:
# Merge, normalize, HVG, PCA
# Note: infection and vaccination annotated separately — inner join yields only 462 shared genes
combined_inf = ad.concat(
    list(adatas_inf.values()),
    join='inner',
    label='study_id',
    keys=list(adatas_inf.keys())
)
combined_inf.obs_names_make_unique()
print(f'Combined infection: {combined_inf.shape}')

combined_inf.layers['counts'] = combined_inf.X.copy()
sc.pp.normalize_total(combined_inf, target_sum=1e4)
sc.pp.log1p(combined_inf)

# HVG selection — do NOT subset, keep full gene matrix in adata.X
sc.pp.highly_variable_genes(combined_inf, n_top_genes=2000, batch_key='study_id')
sc.tl.pca(combined_inf, n_comps=50, use_highly_variable=True)

In [ ]:
# Harmony batch correction
# Note: use hm.run_harmony directly — sc.external.pp.harmony_integrate is unreliable
# ho.Z_corr shape is already (n_cells, 50) — do NOT transpose
ho = hm.run_harmony(combined_inf.obsm['X_pca'], combined_inf.obs, 'study_id')
combined_inf.obsm['X_pca_harmony'] = ho.Z_corr

# Neighbors, UMAP, Leiden
# Resolution 0.3 selected to provide sufficient cluster granularity for CellTypist majority voting
sc.pp.neighbors(combined_inf, use_rep='X_pca_harmony', n_pcs=30)
sc.tl.umap(combined_inf)
sc.tl.leiden(combined_inf, resolution=0.3)
print(f'Leiden clusters: {combined_inf.obs["leiden"].nunique()}')

In [ ]:
# Doublet detection
scrub_inf = scr.Scrublet(combined_inf.layers['counts'])
doublet_scores, predicted_doublets = scrub_inf.scrub_doublets()
combined_inf.obs['doublet_score'] = doublet_scores
combined_inf.obs['predicted_doublet'] = predicted_doublets
print(f'Infection doublets: {predicted_doublets.sum()} ({predicted_doublets.mean()*100:.1f}%)')

In [ ]:
# CellTypist annotation
predictions_inf = celltypist.annotate(combined_inf, model=model_ct, majority_voting=True)
combined_inf = predictions_inf.to_adata()

# Map to challenge ontology
combined_inf.obs['challenge_label'] = combined_inf.obs['challenge_label'].astype(str)
combined_inf.obs['challenge_label'] = combined_inf.obs['majority_voting'].map(label_mapping)
unmapped = combined_inf.obs[combined_inf.obs['challenge_label'].isna()]['majority_voting'].unique()
print('Unmapped labels:', unmapped)

# Apply doublet labels
combined_inf.obs['challenge_label'] = combined_inf.obs['challenge_label'].astype(str)
combined_inf.obs.loc[combined_inf.obs['predicted_doublet'] == True, 'challenge_label'] = 'Doublet'

In [ ]:
# Manual label fixes based on marker gene validation

# Cluster 8: Plasma cells -> Plasmablast (marker validation showed plasmablast signature)
combined_inf.obs.loc[combined_inf.obs['leiden'] == '8', 'challenge_label'] = 'Plasmablast'

# Cluster 13: Low quality cells with no marker expression -> Doublet
combined_inf.obs.loc[combined_inf.obs['leiden'] == '13', 'challenge_label'] = 'Doublet'

# Cluster 14: Multi-lineage marker expression (HBB, HBA, PPBP, CD14) -> Doublet
combined_inf.obs.loc[combined_inf.obs['leiden'] == '14', 'challenge_label'] = 'RBC'

# Intermediate Monocyte recovery: CellTypist collapsed all monocytes to Classical
# Sub-cluster leiden cluster 1 to recover HLA-DRA+ intermediate population
cluster1_cells = combined_inf[combined_inf.obs['leiden'] == '1'].copy()
sc.tl.leiden(cluster1_cells, resolution=0.3, key_added='leiden_sub')
# Sub-cluster 1 (orange) corresponds to HLA-DRA+ Intermediate Monocytes
intermediate_barcodes = cluster1_cells.obs[cluster1_cells.obs['leiden_sub'] == '1'].index
combined_inf.obs.loc[intermediate_barcodes, 'challenge_label'] = 'Intermediate Monocyte'

print(combined_inf.obs['challenge_label'].value_counts())

In [ ]:
# Validation UMAPs
sc.pl.umap(combined_inf, color='majority_voting', legend_fontsize=6,
           title='Infection — CellTypist majority voting', save='_inf_majority_voting.png')
sc.pl.umap(combined_inf, color='challenge_label', legend_fontsize=6,
           title='Infection — challenge labels', save='_inf_challenge_labels.png')

In [ ]:
# Marker gene dotplot validation
marker_genes = {
    'Platelet':                          ['PPBP', 'PF4', 'GNG11'],
    'RBC':                               ['GYPA', 'HBB', 'HBA1'],
    'NK Cell':                           ['NKG7', 'KLRD1', 'GNLY', 'PRF1', 'GZMB'],
    'CD4 Naive / T Central Memory':      ['TCF7', 'LEF1', 'CCR7', 'IL7R', 'CD4'],
    'CD4 T Effector Memory':             ['CCL5', 'GZMK', 'KLRB1', 'IL32', 'AQP3'],
    'Treg':                              ['FOXP3', 'IL2RA', 'TIGIT', 'CTLA4', 'IKZF2'],
    'CD8 Naive / T Central Memory':      ['CD8B', 'CD8A', 'CCR7', 'LEF1', 'S100B'],
    'CD8 Cytotoxic / T Effector Memory': ['GZMH', 'CST7', 'GZMK', 'KLRD1', 'TRGC2'],
    'MAIT Cell':                         ['SLC4A10', 'CXCR6', 'NCR3', 'KLRB1'],
    'Naive B Cell':                      ['IGHM', 'IGHD', 'MS4A1', 'CD79A'],
    'Memory B Cell':                     ['COCH', 'AIM2', 'BANK1', 'SSPN'],
    'Plasmablast':                       ['PRDM1', 'IRF4', 'JCHAIN', 'MZB1', 'MKI67'],
    'Classical Monocyte':                ['CD14', 'S100A8', 'S100A9', 'LYZ', 'FCN1'],
    'Non-Classical Monocyte':            ['FCGR3A', 'CX3CR1', 'LILRB2', 'SIGLEC10'],
    'Intermediate Monocyte':             ['HLA-DRA', 'CCR2', 'CTSS', 'NEAT1'],
    'Plasmacytoid DC':                   ['LILRA4', 'IL3RA', 'SPIB', 'SMPD3', 'ITM2C'],
    'Conventional DC 1':                 ['CLEC9A', 'XCR1', 'CADM1', 'FLT3'],
    'Conventional DC 2':                 ['FCER1A', 'CLEC10A', 'CD1C', 'HLA-DQA1'],
}

var_names = combined_inf.var_names.tolist()
marker_genes_filtered = {ct: [g for g in genes if g in var_names] for ct, genes in marker_genes.items()}

sc.pl.dotplot(combined_inf, var_names=marker_genes_filtered, groupby='challenge_label',
              dendrogram=False, standard_scale='var', figsize=(20, 8), save='_inf_markers_final.png')

In [ ]:
# Export per-study TSVs
os.makedirs('annotations', exist_ok=True)
for study in combined_inf.obs['study_id'].unique():
    subset = combined_inf.obs[combined_inf.obs['study_id'] == study][['challenge_label', 'conf_score']].reset_index()
    subset.columns = ['cell_barcode', 'predicted_cell_type', 'confidence_score']
    subset.to_csv(f'annotations/{study}_annotation.tsv', sep='\t', index=False)
    print(f'Wrote annotations/{study}_annotation.tsv ({len(subset)} cells)')

# Save object
combined_inf.obs['age'] = combined_inf.obs['age'].astype(str)
combined_inf.write_h5ad('objects/combined_infection.h5ad')
print('Saved combined_infection.h5ad')

## 4. Vaccination Cohort (studies 01, 02, 07, 09)

Study 03 excluded from combined object due to h5ad format incompatibility — processed separately (Section 5).  
Study 09 is CITE-seq; protein features removed, RNA only retained.

In [ ]:
# Load corrected h5ad files
vacc01 = sc.read_h5ad('vaccination_study_01/vaccination_study_01_corrected.h5ad')
vacc02 = sc.read_h5ad('vaccination_study_02/vaccination_study_02_corrected.h5ad')
vacc07 = sc.read_h5ad('vaccination_study_07/vaccination_study_07_corrected.h5ad')
vacc09 = sc.read_h5ad('vaccination_study_09/vaccination_study_09_corrected.h5ad')

# Study 09: remove protein features, keep RNA only
rna_mask = ~vacc09.var_names.str.contains('PROT')
vacc09 = vacc09[:, rna_mask].copy()
print(f'Study 09 after RNA filter: {vacc09.n_vars} genes')

adatas_vacc = {
    'vaccination_study_01': vacc01,
    'vaccination_study_02': vacc02,
    'vaccination_study_07': vacc07,
    'vaccination_study_09': vacc09,
}

In [ ]:
# QC per study
for name in list(adatas_vacc.keys()):
    adatas_vacc[name] = filter_study(adatas_vacc[name], name)

In [ ]:
# Merge, normalize, HVG, PCA
combined_vacc = ad.concat(
    list(adatas_vacc.values()),
    join='inner',
    label='study_id',
    keys=list(adatas_vacc.keys())
)
combined_vacc.obs_names_make_unique()
print(f'Combined vaccination: {combined_vacc.shape}')

combined_vacc.layers['counts'] = combined_vacc.X.copy()
sc.pp.normalize_total(combined_vacc, target_sum=1e4)
sc.pp.log1p(combined_vacc)

# HVG selection — do NOT subset, keep full gene matrix in adata.X
sc.pp.highly_variable_genes(combined_vacc, n_top_genes=2000, batch_key='study_id')
sc.tl.pca(combined_vacc, n_comps=50, use_highly_variable=True)

In [ ]:
# Harmony batch correction
ho = hm.run_harmony(combined_vacc.obsm['X_pca'], combined_vacc.obs, 'study_id')
combined_vacc.obsm['X_pca_harmony'] = ho.Z_corr

sc.pp.neighbors(combined_vacc, use_rep='X_pca_harmony', n_pcs=30)
sc.tl.umap(combined_vacc)
sc.tl.leiden(combined_vacc, resolution=0.3)
print(f'Leiden clusters: {combined_vacc.obs["leiden"].nunique()}')

In [ ]:
# Doublet detection
scrub_vacc = scr.Scrublet(combined_vacc.layers['counts'])
doublet_scores, predicted_doublets = scrub_vacc.scrub_doublets()
combined_vacc.obs['doublet_score'] = doublet_scores
combined_vacc.obs['predicted_doublet'] = predicted_doublets
print(f'Vaccination doublets: {predicted_doublets.sum()} ({predicted_doublets.mean()*100:.1f}%)')

In [ ]:
# CellTypist annotation
predictions_vacc = celltypist.annotate(combined_vacc, model=model_ct, majority_voting=True)
combined_vacc = predictions_vacc.to_adata()

# Map to challenge ontology
combined_vacc.obs['challenge_label'] = combined_vacc.obs['majority_voting'].map(label_mapping)
unmapped = combined_vacc.obs[combined_vacc.obs['challenge_label'].isna()]['majority_voting'].unique()
print('Unmapped labels:', unmapped)

# Apply doublet and manual fixes
combined_vacc.obs['challenge_label'] = combined_vacc.obs['challenge_label'].astype(str)
combined_vacc.obs.loc[combined_vacc.obs['predicted_doublet'] == True, 'challenge_label'] = 'Doublet'

# Cluster 15: Plasma cells -> Plasmablast
combined_vacc.obs.loc[combined_vacc.obs['leiden'] == '15', 'challenge_label'] = 'Plasmablast'

print(combined_vacc.obs['challenge_label'].value_counts())

In [ ]:
# Validation UMAPs and dotplot
sc.pl.umap(combined_vacc, color='majority_voting', legend_fontsize=6,
           title='Vaccination — CellTypist majority voting', save='_vacc_majority_voting.png')
sc.pl.umap(combined_vacc, color='challenge_label', legend_fontsize=6,
           title='Vaccination — challenge labels', save='_vacc_challenge_labels.png')

var_names = combined_vacc.var_names.tolist()
marker_genes_filtered_vacc = {ct: [g for g in genes if g in var_names] for ct, genes in marker_genes.items()}
sc.pl.dotplot(combined_vacc, var_names=marker_genes_filtered_vacc, groupby='challenge_label',
              dendrogram=False, standard_scale='var', figsize=(20, 8), save='_vacc_markers_final.png')

In [ ]:
# Export per-study TSVs
for study in combined_vacc.obs['study_id'].unique():
    subset = combined_vacc.obs[combined_vacc.obs['study_id'] == study][['challenge_label', 'conf_score']].reset_index()
    subset.columns = ['cell_barcode', 'predicted_cell_type', 'confidence_score']
    subset.to_csv(f'annotations/{study}_annotation.tsv', sep='\t', index=False)
    print(f'Wrote annotations/{study}_annotation.tsv ({len(subset)} cells)')

combined_vacc.obs['age'] = combined_vacc.obs['age'].astype(str)
combined_vacc.write_h5ad('objects/combined_vaccination.h5ad')
print('Saved combined_vaccination.h5ad')

## 5. Vaccination Study 03 (standalone)

h5ad format incompatible with available anndata version. Loaded from MTX files.  
Tissue is myeloid-enriched (monocytes and dendritic cells). No batch correction applied —  
UMAP by timepoint showed no meaningful batch effect across donors.

In [ ]:
# Load from MTX
study03 = sc.read_mtx('vaccination_study_03/vaccination_study_03.mtx').T
barcodes = pd.read_csv('vaccination_study_03/vaccination_study_03_barcodes.tsv', header=None, sep='\t')[0].values
features = pd.read_csv('vaccination_study_03/vaccination_study_03_features.tsv', header=None, sep='\t')[0].values
study03.obs_names = barcodes
study03.var_names = features

# Merge metadata
meta = pd.read_csv('vaccination_study_03/vaccination_study_03_metadata.csv', index_col=0)
shared_cols = ['sample_id', 'subject_id', 'arm', 'timepoint', 'tissue', 'age']
study03.obs = study03.obs.join(meta[shared_cols], how='left')
study03.obs['study_id'] = 'vaccination_study_03'

print(f'Study 03: {study03.shape}')
print(f'Expression range: {study03.X.min():.2f} to {study03.X.max():.2f}')

In [ ]:
# QC
study03.var['mt'] = study03.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(study03, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
n_before = study03.n_obs
study03 = study03[
    (study03.obs['n_genes_by_counts'] >= 200) &
    (study03.obs['n_genes_by_counts'] <= 6000) &
    (study03.obs['total_counts'] < 50000) &
    (study03.obs['pct_counts_mt'] < 15)
].copy()
print(f'{n_before} -> {study03.n_obs} cells after QC')

# Normalize
study03.layers['counts'] = study03.X.copy()
sc.pp.normalize_total(study03, target_sum=1e4)
sc.pp.log1p(study03)

In [ ]:
# HVG, PCA, neighbors, UMAP
# No Harmony needed — timepoint UMAP showed no batch effect
sc.pp.highly_variable_genes(study03, n_top_genes=2000)
sc.tl.pca(study03, n_comps=50, use_highly_variable=True)
sc.pp.neighbors(study03, n_pcs=30)
sc.tl.umap(study03)

# Resolution selection via silhouette and Calinski-Harabasz sweep
# Both metrics peaked at 0.1, consistent with myeloid-enriched dataset (~6-7 expected cell types)
sc.tl.leiden(study03, resolution=0.1)

In [ ]:
# CellTypist annotation
predictions_03 = celltypist.annotate(study03, model=model_ct, majority_voting=True)
study03 = predictions_03.to_adata()

study03.obs['challenge_label'] = study03.obs['majority_voting'].map(label_mapping)
# Relabel Plasma Cell -> Plasmablast
study03.obs['challenge_label'] = study03.obs['challenge_label'].astype(str)
study03.obs.loc[study03.obs['challenge_label'] == 'Plasma Cell', 'challenge_label'] = 'Plasmablast'

unmapped = study03.obs[study03.obs['challenge_label'].isna()]['majority_voting'].unique()
print('Unmapped:', unmapped)
print(study03.obs['challenge_label'].value_counts())

In [ ]:
# Validation UMAP and dotplot
sc.pl.umap(study03, color='challenge_label', legend_fontsize=6,
           title='Study 03 — challenge labels', save='_study03_challenge_labels.png')

var_names_03 = study03.var_names.tolist()
marker_genes_filtered_03 = {ct: [g for g in genes if g in var_names_03] for ct, genes in marker_genes.items()}
sc.pl.dotplot(study03, var_names=marker_genes_filtered_03, groupby='challenge_label',
              dendrogram=False, standard_scale='var', figsize=(20, 8), save='_study03_markers.png')

# Export TSV
study03.obs[['challenge_label', 'conf_score']].reset_index().rename(
    columns={'index': 'cell_barcode', 'challenge_label': 'predicted_cell_type', 'conf_score': 'confidence_score'}
)[['cell_barcode', 'predicted_cell_type', 'confidence_score']].to_csv(
    'annotations/vaccination_study_03_annotation.tsv', sep='\t', index=False
)
print('Wrote annotations/vaccination_study_03_annotation.tsv')

study03.obs['age'] = study03.obs['age'].astype(str)
study03.write_h5ad('objects/vaccination_study_03.h5ad')
print('Saved vaccination_study_03.h5ad')

## 6. KNN Purity Validation

In [ ]:
def knn_purity(adata, embedding_key='X_pca_harmony', label_key='challenge_label', n_neighbors=15, sample=10000):
    idx = np.random.choice(adata.n_obs, sample, replace=False)
    X = adata.obsm[embedding_key][idx]
    labels = np.array(adata.obs[label_key].values[idx])
    nn = NearestNeighbors(n_neighbors=n_neighbors+1).fit(X)
    _, indices = nn.kneighbors(X)
    neighbor_labels = labels[indices[:, 1:]]
    majority_label = stats.mode(neighbor_labels, axis=1).mode.flatten()
    purity = np.mean(labels == majority_label)
    print(f'{label_key} KNN purity (n={sample}): {purity:.3f}')
    return purity

knn_purity(combined_inf)
knn_purity(combined_vacc)
knn_purity(study03, embedding_key='X_pca')

## 7. Ontology Label Validation

In [ ]:
valid_terminal = {
    'Platelet', 'RBC', 'HSC', 'Doublet', 'NK Cell',
    'CD4 Naive / T Central Memory', 'CD4 T Effector Memory', 'Treg',
    'CD8 Naive / T Central Memory', 'CD8 Cytotoxic / T Effector Memory',
    'gdT Cell', 'MAIT Cell', 'NKT Cell',
    'Plasma Cell', 'Plasmablast', 'Naive B Cell', 'Memory B Cell',
    'Classical Monocyte', 'Non-Classical Monocyte', 'Intermediate Monocyte',
    'Neutrophil', 'Eosinophil', 'Basophil', 'Mast Cell',
    'Plasmacytoid DC', 'Conventional DC 1', 'Conventional DC 2'
}
valid_parent = {
    'Blood Cell', 'Leukocyte', 'Lymphoid Cell', 'T Cell', 'CD4 T Cell (ab)',
    'CD8 T Cell (ab)', 'B Cell', 'Effector B', 'Myeloid Cell', 'Monocyte',
    'Granulocyte', 'DC'
}
valid_all = valid_terminal | valid_parent

for name, adata in [('infection', combined_inf), ('vaccination', combined_vacc), ('study03', study03)]:
    invalid = set(adata.obs['challenge_label'].unique()) - valid_all
    print(f'{name} invalid labels: {invalid}')